In [0]:
%pip install unidecode

Note: you may need to restart the kernel using %restart_python or dbutils.library.restartPython() to use updated packages.


In [0]:
from pyspark.sql.functions import *
from unidecode import unidecode
from itertools import chain
import re

In [0]:
# Func para descartar colunas nao populadas
def drop_null_columns(df):
    # 1. Contar valores nulos por coluna
    null_counts = (
        df
        .select(
            [count(when(col(c).isNull(), c)).alias(c) for c in df.columns]
        )
        .collect()[0]
        .asDict()
    )

    # 2. Obter o total de linhas
    total_rows = df.count()

    # 3. Identificar colunas 100% nulas
    columns_to_drop = [k for k, v in null_counts.items() if v == total_rows]

    # 4. Remover as colunas 100% nulas
    return df.drop(*columns_to_drop)

# ETL

## Extração

In [0]:
df_pm_22 = spark.read.csv("/Volumes/workspace/default/passos_magicos/BASE DE DADOS PEDE 2024 - DATATHON - PEDE2022.csv", header=True)

df_pm_23 = spark.read.csv("/Volumes/workspace/default/passos_magicos/BASE DE DADOS PEDE 2024 - DATATHON - PEDE2023.csv", header=True)

df_pm_24 = spark.read.csv("/Volumes/workspace/default/passos_magicos/BASE DE DADOS PEDE 2024 - DATATHON - PEDE2024.csv", header=True)

## Transformação

### Normalizando Estrutura de Dados

Normalizando nomes de colunas entre DFs

In [0]:
dict_cols_22 = {
    "Nome": "Nome Anonimizado"
    ,"Idade 22": "Idade"
    ,"Matem": "Mat"
    ,"Portug": "Por"
    ,"Inglês": "Ing"
    ,"Defas": "Defasagem"
}

df_pm_22 = (
    df_pm_22
    .withColumnsRenamed(dict_cols_22)
    .withColumn("Ano", lit("2022-01-01"))
    .withColumn("Data de Nasc", concat(col("Ano nasc"), lit("-01-01")))
    .withColumn("Cg", floor(translate(col("Cg"), ",", ".")))
    .drop("Ano nasc")
)

df_pm_22.limit(5).display()

RA,Fase,Turma,Nome Anonimizado,Idade,Gênero,Ano ingresso,Instituição de ensino,Pedra 20,Pedra 21,Pedra 22,INDE 22,Cg,Cf,Ct,Nº Av,Avaliador1,Rec Av1,Avaliador2,Rec Av2,Avaliador3,Rec Av3,Avaliador4,Rec Av4,IAA,IEG,IPS,Rec Psicologia,IDA,Mat,Por,Ing,Indicado,Atingiu PV,IPV,IAN,Fase ideal,Defasagem,Destaque IEG,Destaque IDA,Destaque IPV,Ano,Data de Nasc
RA-1,7,A,Aluno-1,19,Menina,2016,Escola Pública,Ametista,Ametista,Quartzo,"5,783",753,18,10,4,Avaliador-5,Mantido na Fase atual,Avaliador-27,Promovido de Fase + Bolsa,Avaliador-28,Promovido de Fase,Avaliador-31,Mantido na Fase atual,"8,3","4,1","5,6",Requer avaliação,"4,0","2,7","3,5","6,0",Sim,Não,"7,278","5,000",Fase 8 (Universitários),-1,Melhorar: Melhorar a sua entrega de lições de casa.,Melhorar: Empenhar-se mais nas aulas e avaliações.,Melhorar: Integrar-se mais aos Princípios Passos Mágicos.,2022-01-01,2003-01-01
RA-2,7,A,Aluno-2,17,Menina,2017,Rede Decisão,Ametista,Ametista,Ametista,"7,055",469,8,3,4,Avaliador-5,Promovido de Fase,Avaliador-27,Promovido de Fase + Bolsa,Avaliador-28,Promovido de Fase,Avaliador-31,Promovido de Fase + Bolsa,"8,8","5,2","6,3",Sem limitações,"6,8","6,3","4,5","9,7",Não,Não,"6,778","10,000",Fase 7 (3º EM),0,Melhorar: Melhorar a sua entrega de lições de casa.,Melhorar: Empenhar-se mais nas aulas e avaliações.,Melhorar: Integrar-se mais aos Princípios Passos Mágicos.,2022-01-01,2005-01-01
RA-3,7,A,Aluno-3,17,Menina,2016,Rede Decisão,Ametista,Ametista,Ágata,"6,591",629,13,6,4,Avaliador-5,Promovido de Fase,Avaliador-27,Promovido de Fase + Bolsa,Avaliador-28,Promovido de Fase + Bolsa,Avaliador-31,Promovido de Fase + Bolsa,"0,0","7,9","5,6",Sem limitações,"5,6","5,8","4,0","6,9",Não,Não,"7,556","10,000",Fase 7 (3º EM),0,Destaque: A sua boa entrega das lições de casa.,Melhorar: Empenhar-se mais nas aulas e avaliações.,Destaque: A sua boa integração aos Princípios Passos Mágicos.,2022-01-01,2005-01-01
RA-4,7,A,Aluno-4,17,Menino,2017,Rede Decisão,Ametista,Ametista,Quartzo,"5,951",731,15,7,4,Avaliador-5,Promovido de Fase,Avaliador-27,Mantido na Fase atual,Avaliador-28,Mantido na Fase atual,Avaliador-31,Mantido na Fase atual,"8,8","4,5","5,6",Requer avaliação,"5,0","2,8","3,5","8,7",Não,Não,"5,278","10,000",Fase 7 (3º EM),0,Melhorar: Melhorar a sua entrega de lições de casa.,Melhorar: Empenhar-se mais nas aulas e avaliações.,Melhorar: Integrar-se mais aos Princípios Passos Mágicos.,2022-01-01,2005-01-01
RA-5,7,A,Aluno-5,17,Menina,2016,Rede Decisão,Ametista,Ametista,Ametista,"7,427",344,6,2,4,Avaliador-5,Promovido de Fase,Avaliador-27,Promovido de Fase + Bolsa,Avaliador-28,Promovido de Fase + Bolsa,Avaliador-31,Promovido de Fase + Bolsa,"7,9","8,6","5,6",Requer avaliação,"5,2","7,0","2,9","5,7",Não,Não,"7,389","10,000",Fase 7 (3º EM),0,Destaque: A sua boa entrega das lições de casa.,Melhorar: Empenhar-se mais nas aulas e avaliações.,Melhorar: Integrar-se mais aos Princípios Passos Mágicos.,2022-01-01,2005-01-01


In [0]:
dict_cols_23 = {
    "INDE 2023": "INDE 23"
    ,"Pedra 2023": "Pedra 23"
}

# Drop de colunas 100% nulas
df_pm_23 = drop_null_columns(df_pm_23)

df_pm_23 = (
    df_pm_23
    .withColumnsRenamed(dict_cols_23)
    .withColumn("Ano", lit("2023-01-01"))
    .withColumn("Data de Nasc", date_format(to_date(col("Data de Nasc"), "M/d/yyyy"), "yyyy-MM-dd"))
    .withColumn("Idade", when(col("Idade").try_cast("int").isNotNull(), col("Idade")).otherwise(floor(months_between(lit("2023-12-31"), col("Data de Nasc")) / 12)))
)

df_pm_23.limit(5).display()

RA,Fase,INDE 23,Pedra 23,Turma,Nome Anonimizado,Data de Nasc,Idade,Gênero,Ano ingresso,Instituição de ensino,Pedra 20,Pedra 21,Pedra 22,INDE 22,Nº Av,Avaliador1,Avaliador2,Avaliador3,Avaliador4,IAA,IEG,IPS,IPP,IDA,Mat,Por,Ing,IPV,IAN,Fase Ideal,Defasagem,Ano
RA-861,ALFA,"9,31095",Topázio,ALFA A - G0/G1,Aluno-861,2015-06-17,8,Feminino,2023,Pública,null,null,null,null,2,Avaliador-11,Avaliador-2,null,null,"9,5","10,0","8,13","8,4375","9,6","9,8","9,4",null,"8,92",10,ALFA (1° e 2° ano),0,2023-01-01
RA-862,ALFA,"8,2212",Topázio,ALFA A - G0/G1,Aluno-862,2014-05-31,9,Masculino,2023,Pública,null,null,null,null,2,Avaliador-11,Avaliador-2,null,null,"8,5","9,1","8,14","7,5","8,9","8,5","9,2",null,"8,585",5,Fase 1 (3° e 4° ano),-1,2023-01-01
RA-863,ALFA,"5,92975",Quartzo,ALFA A - G0/G1,Aluno-863,2016-02-25,7,Masculino,2023,Pública,null,null,null,null,2,Avaliador-11,Avaliador-2,null,null,"0,0","7,6","3,14","5,9375","6,3","7,0","5,5",null,"6,26",10,ALFA (1° e 2° ano),0,2023-01-01
RA-864,ALFA,"7,034",Ametista,ALFA A - G0/G1,Aluno-864,2015-12-03,8,Feminino,2023,Pública,null,null,null,null,2,Avaliador-11,Avaliador-2,null,null,"0,0","7,6","8,14","7,5","6,3","7,0","5,5",null,"8,5",10,ALFA (1° e 2° ano),0,2023-01-01
RA-865,ALFA,"8,1552",Topázio,ALFA A - G0/G1,Aluno-865,2014-11-13,8,Masculino,2023,Pública,null,null,null,null,2,Avaliador-11,Avaliador-2,null,null,"8,5","8,7","7,52","7,5","7,4","7,3","7,5",null,"7,915",10,ALFA (1° e 2° ano),0,2023-01-01


In [0]:
dict_cols_24 = {
    "INDE 2024": "INDE 24"
    ,"Pedra 2024": "Pedra 24"
}

# Drop de colunas 100% nulas
df_pm_24 = drop_null_columns(df_pm_24)

df_pm_24 = (
    df_pm_24
    .withColumnsRenamed(dict_cols_24)
    .withColumn("Ano", lit("2024-01-01"))
    .withColumn("Data de Nasc", date_format(to_date(col("Data de Nasc"), "dd/MM/yyyy"), "yyyy-MM-dd"))
    .withColumn("Ativo Inativo", coalesce(col("Ativo/ Inativo48"), col("Ativo/ Inativo49")))
    .drop("Ativo/ Inativo48", "Ativo/ Inativo49")
)

df_pm_24.limit(5).display()

RA,Fase,INDE 24,Pedra 24,Turma,Nome Anonimizado,Data de Nasc,Idade,Gênero,Ano ingresso,Instituição de ensino,Pedra 20,Pedra 21,Pedra 22,Pedra 23,INDE 22,INDE 23,Nº Av,Avaliador1,Avaliador2,Avaliador3,Avaliador4,Avaliador5,Avaliador6,IAA,IEG,IPS,IPP,IDA,Mat,Por,Ing,IPV,IAN,Fase Ideal,Defasagem,Escola,Ano,Ativo Inativo
RA-1275,ALFA,"7,611366667",Ametista,ALFA A - G0/G1,Aluno-1275,2016-07-28,8,Masculino,2024,Pública,null,null,null,null,null,null,3,Avaliador-11,Avaliador-2,Avaliador-9,null,null,null,"10,0","8,7","6,3","5,6","8,0","10,0","6,0",null,"5,4",10,ALFA (1° e 2° ano),0,EE Chácara Florida II,2024-01-01,Cursando
RA-1276,ALFA,"8,002866667",Topázio,ALFA A - G0/G1,Aluno-1276,2016-10-16,8,Feminino,2024,Pública,null,null,null,null,null,null,3,Avaliador-11,Avaliador-2,Avaliador-9,null,null,null,"10,0","9,3","3,8","7,5","8,0","10,0","6,0",null,"7,1",10,ALFA (1° e 2° ano),0,EE Chácara Florida II,2024-01-01,Cursando
RA-1277,ALFA,"7,9522",Ametista,ALFA A - G0/G1,Aluno-1277,2016-08-16,8,Masculino,2024,Pública,null,null,null,null,null,null,3,Avaliador-11,Avaliador-2,Avaliador-9,null,null,null,"10,0","9,1","3,8","7,5","8,0","10,0","6,0",null,"7,0",10,ALFA (1° e 2° ano),0,EE Dom Pedro Villas Boas de Souza,2024-01-01,Cursando
RA-868,ALFA,"7,156366667",Ametista,ALFA A - G0/G1,Aluno-868,2015-11-08,8,Masculino,2023,Pública,null,null,null,Topázio,null,"8,63895",3,Avaliador-11,Avaliador-2,Avaliador-9,null,null,null,"8,0","9,8","3,8","6,9","7,0","8,0","6,0",null,"7,2",5,Fase 1 (3° e 4° ano),-1,EE Chácara Florida II,2024-01-01,Cursando
RA-1278,ALFA,"5,4442",Quartzo,ALFA A - G0/G1,Aluno-1278,2015-03-22,9,Masculino,2024,Pública,null,null,null,null,null,null,3,Avaliador-11,Avaliador-2,Avaliador-9,null,null,null,"9,0","4,2","3,8","5,0","7,5","8,0","7,0",null,"4,2",5,Fase 1 (3° e 4° ano),-1,EM Etelvina Delfim Simões,2024-01-01,Cursando


### Unificando Estrutura

In [0]:
df_pm = (
    df_pm_22
    .unionByName(df_pm_23, allowMissingColumns=True)
    .unionByName(df_pm_24, allowMissingColumns=True)
)

df_pm.limit(5).display()

RA,Fase,Turma,Nome Anonimizado,Idade,Gênero,Ano ingresso,Instituição de ensino,Pedra 20,Pedra 21,Pedra 22,INDE 22,Cg,Cf,Ct,Nº Av,Avaliador1,Rec Av1,Avaliador2,Rec Av2,Avaliador3,Rec Av3,Avaliador4,Rec Av4,IAA,IEG,IPS,Rec Psicologia,IDA,Mat,Por,Ing,Indicado,Atingiu PV,IPV,IAN,Fase ideal,Defasagem,Destaque IEG,Destaque IDA,Destaque IPV,Ano,Data de Nasc,INDE 23,Pedra 23,IPP,INDE 24,Pedra 24,Avaliador5,Avaliador6,Escola,Ativo Inativo
RA-1,7,A,Aluno-1,19,Menina,2016,Escola Pública,Ametista,Ametista,Quartzo,"5,783",753,18,10,4,Avaliador-5,Mantido na Fase atual,Avaliador-27,Promovido de Fase + Bolsa,Avaliador-28,Promovido de Fase,Avaliador-31,Mantido na Fase atual,"8,3","4,1","5,6",Requer avaliação,"4,0","2,7","3,5","6,0",Sim,Não,"7,278","5,000",Fase 8 (Universitários),-1,Melhorar: Melhorar a sua entrega de lições de casa.,Melhorar: Empenhar-se mais nas aulas e avaliações.,Melhorar: Integrar-se mais aos Princípios Passos Mágicos.,2022-01-01,2003-01-01,null,null,null,null,null,null,null,null,null
RA-2,7,A,Aluno-2,17,Menina,2017,Rede Decisão,Ametista,Ametista,Ametista,"7,055",469,8,3,4,Avaliador-5,Promovido de Fase,Avaliador-27,Promovido de Fase + Bolsa,Avaliador-28,Promovido de Fase,Avaliador-31,Promovido de Fase + Bolsa,"8,8","5,2","6,3",Sem limitações,"6,8","6,3","4,5","9,7",Não,Não,"6,778","10,000",Fase 7 (3º EM),0,Melhorar: Melhorar a sua entrega de lições de casa.,Melhorar: Empenhar-se mais nas aulas e avaliações.,Melhorar: Integrar-se mais aos Princípios Passos Mágicos.,2022-01-01,2005-01-01,null,null,null,null,null,null,null,null,null
RA-3,7,A,Aluno-3,17,Menina,2016,Rede Decisão,Ametista,Ametista,Ágata,"6,591",629,13,6,4,Avaliador-5,Promovido de Fase,Avaliador-27,Promovido de Fase + Bolsa,Avaliador-28,Promovido de Fase + Bolsa,Avaliador-31,Promovido de Fase + Bolsa,"0,0","7,9","5,6",Sem limitações,"5,6","5,8","4,0","6,9",Não,Não,"7,556","10,000",Fase 7 (3º EM),0,Destaque: A sua boa entrega das lições de casa.,Melhorar: Empenhar-se mais nas aulas e avaliações.,Destaque: A sua boa integração aos Princípios Passos Mágicos.,2022-01-01,2005-01-01,null,null,null,null,null,null,null,null,null
RA-4,7,A,Aluno-4,17,Menino,2017,Rede Decisão,Ametista,Ametista,Quartzo,"5,951",731,15,7,4,Avaliador-5,Promovido de Fase,Avaliador-27,Mantido na Fase atual,Avaliador-28,Mantido na Fase atual,Avaliador-31,Mantido na Fase atual,"8,8","4,5","5,6",Requer avaliação,"5,0","2,8","3,5","8,7",Não,Não,"5,278","10,000",Fase 7 (3º EM),0,Melhorar: Melhorar a sua entrega de lições de casa.,Melhorar: Empenhar-se mais nas aulas e avaliações.,Melhorar: Integrar-se mais aos Princípios Passos Mágicos.,2022-01-01,2005-01-01,null,null,null,null,null,null,null,null,null
RA-5,7,A,Aluno-5,17,Menina,2016,Rede Decisão,Ametista,Ametista,Ametista,"7,427",344,6,2,4,Avaliador-5,Promovido de Fase,Avaliador-27,Promovido de Fase + Bolsa,Avaliador-28,Promovido de Fase + Bolsa,Avaliador-31,Promovido de Fase + Bolsa,"7,9","8,6","5,6",Requer avaliação,"5,2","7,0","2,9","5,7",Não,Não,"7,389","10,000",Fase 7 (3º EM),0,Destaque: A sua boa entrega das lições de casa.,Melhorar: Empenhar-se mais nas aulas e avaliações.,Melhorar: Integrar-se mais aos Princípios Passos Mágicos.,2022-01-01,2005-01-01,null,null,null,null,null,null,null,null,null


### Transformando Estrutura

In [0]:
list_pm_cols = df_pm.columns

#Remover acentos
list_pm_renamed_cols = [unidecode(i) for i in list_pm_cols]

#Remover caracteres especiais
list_pm_renamed_cols = [re.sub(r"[^a-zA-Z0-9 ]", "", i) for i in list_pm_renamed_cols]

#Trocar espacos por underline
list_pm_renamed_cols = [i.replace(" ", "_") for i in list_pm_renamed_cols]

dict_pm_cols = dict(zip(list_pm_cols, list_pm_renamed_cols))

print(dict_pm_cols)

#Renoear colunas
df_pm = df_pm.withColumnsRenamed(dict_pm_cols)

df_pm.limit(5).display()

{'RA': 'RA', 'Fase': 'Fase', 'Turma': 'Turma', 'Nome Anonimizado': 'Nome_Anonimizado', 'Idade': 'Idade', 'Gênero': 'Genero', 'Ano ingresso': 'Ano_ingresso', 'Instituição de ensino': 'Instituicao_de_ensino', 'Pedra 20': 'Pedra_20', 'Pedra 21': 'Pedra_21', 'Pedra 22': 'Pedra_22', 'INDE 22': 'INDE_22', 'Cg': 'Cg', 'Cf': 'Cf', 'Ct': 'Ct', 'Nº Av': 'No_Av', 'Avaliador1': 'Avaliador1', 'Rec Av1': 'Rec_Av1', 'Avaliador2': 'Avaliador2', 'Rec Av2': 'Rec_Av2', 'Avaliador3': 'Avaliador3', 'Rec Av3': 'Rec_Av3', 'Avaliador4': 'Avaliador4', 'Rec Av4': 'Rec_Av4', 'IAA': 'IAA', 'IEG': 'IEG', 'IPS': 'IPS', 'Rec Psicologia': 'Rec_Psicologia', 'IDA': 'IDA', 'Mat': 'Mat', 'Por': 'Por', 'Ing': 'Ing', 'Indicado': 'Indicado', 'Atingiu PV': 'Atingiu_PV', 'IPV': 'IPV', 'IAN': 'IAN', 'Fase ideal': 'Fase_ideal', 'Defasagem': 'Defasagem', 'Destaque IEG': 'Destaque_IEG', 'Destaque IDA': 'Destaque_IDA', 'Destaque IPV': 'Destaque_IPV', 'Ano': 'Ano', 'Data de Nasc': 'Data_de_Nasc', 'INDE 23': 'INDE_23', 'Pedra 23': '

RA,Fase,Turma,Nome_Anonimizado,Idade,Genero,Ano_ingresso,Instituicao_de_ensino,Pedra_20,Pedra_21,Pedra_22,INDE_22,Cg,Cf,Ct,No_Av,Avaliador1,Rec_Av1,Avaliador2,Rec_Av2,Avaliador3,Rec_Av3,Avaliador4,Rec_Av4,IAA,IEG,IPS,Rec_Psicologia,IDA,Mat,Por,Ing,Indicado,Atingiu_PV,IPV,IAN,Fase_ideal,Defasagem,Destaque_IEG,Destaque_IDA,Destaque_IPV,Ano,Data_de_Nasc,INDE_23,Pedra_23,IPP,INDE_24,Pedra_24,Avaliador5,Avaliador6,Escola,Ativo_Inativo
RA-1,7,A,Aluno-1,19,Menina,2016,Escola Pública,Ametista,Ametista,Quartzo,"5,783",753,18,10,4,Avaliador-5,Mantido na Fase atual,Avaliador-27,Promovido de Fase + Bolsa,Avaliador-28,Promovido de Fase,Avaliador-31,Mantido na Fase atual,"8,3","4,1","5,6",Requer avaliação,"4,0","2,7","3,5","6,0",Sim,Não,"7,278","5,000",Fase 8 (Universitários),-1,Melhorar: Melhorar a sua entrega de lições de casa.,Melhorar: Empenhar-se mais nas aulas e avaliações.,Melhorar: Integrar-se mais aos Princípios Passos Mágicos.,2022-01-01,2003-01-01,null,null,null,null,null,null,null,null,null
RA-2,7,A,Aluno-2,17,Menina,2017,Rede Decisão,Ametista,Ametista,Ametista,"7,055",469,8,3,4,Avaliador-5,Promovido de Fase,Avaliador-27,Promovido de Fase + Bolsa,Avaliador-28,Promovido de Fase,Avaliador-31,Promovido de Fase + Bolsa,"8,8","5,2","6,3",Sem limitações,"6,8","6,3","4,5","9,7",Não,Não,"6,778","10,000",Fase 7 (3º EM),0,Melhorar: Melhorar a sua entrega de lições de casa.,Melhorar: Empenhar-se mais nas aulas e avaliações.,Melhorar: Integrar-se mais aos Princípios Passos Mágicos.,2022-01-01,2005-01-01,null,null,null,null,null,null,null,null,null
RA-3,7,A,Aluno-3,17,Menina,2016,Rede Decisão,Ametista,Ametista,Ágata,"6,591",629,13,6,4,Avaliador-5,Promovido de Fase,Avaliador-27,Promovido de Fase + Bolsa,Avaliador-28,Promovido de Fase + Bolsa,Avaliador-31,Promovido de Fase + Bolsa,"0,0","7,9","5,6",Sem limitações,"5,6","5,8","4,0","6,9",Não,Não,"7,556","10,000",Fase 7 (3º EM),0,Destaque: A sua boa entrega das lições de casa.,Melhorar: Empenhar-se mais nas aulas e avaliações.,Destaque: A sua boa integração aos Princípios Passos Mágicos.,2022-01-01,2005-01-01,null,null,null,null,null,null,null,null,null
RA-4,7,A,Aluno-4,17,Menino,2017,Rede Decisão,Ametista,Ametista,Quartzo,"5,951",731,15,7,4,Avaliador-5,Promovido de Fase,Avaliador-27,Mantido na Fase atual,Avaliador-28,Mantido na Fase atual,Avaliador-31,Mantido na Fase atual,"8,8","4,5","5,6",Requer avaliação,"5,0","2,8","3,5","8,7",Não,Não,"5,278","10,000",Fase 7 (3º EM),0,Melhorar: Melhorar a sua entrega de lições de casa.,Melhorar: Empenhar-se mais nas aulas e avaliações.,Melhorar: Integrar-se mais aos Princípios Passos Mágicos.,2022-01-01,2005-01-01,null,null,null,null,null,null,null,null,null
RA-5,7,A,Aluno-5,17,Menina,2016,Rede Decisão,Ametista,Ametista,Ametista,"7,427",344,6,2,4,Avaliador-5,Promovido de Fase,Avaliador-27,Promovido de Fase + Bolsa,Avaliador-28,Promovido de Fase + Bolsa,Avaliador-31,Promovido de Fase + Bolsa,"7,9","8,6","5,6",Requer avaliação,"5,2","7,0","2,9","5,7",Não,Não,"7,389","10,000",Fase 7 (3º EM),0,Destaque: A sua boa entrega das lições de casa.,Melhorar: Empenhar-se mais nas aulas e avaliações.,Melhorar: Integrar-se mais aos Princípios Passos Mágicos.,2022-01-01,2005-01-01,null,null,null,null,null,null,null,null,null


In [0]:
dict_values_to_replace = {
    "Menino": "Masculino"
    ,"Menina": "Feminino"
    ,"Agata": "Ágata"
    ,"#N/A": None
    ,"#DIV/0!": None
    ,"INCLUIR": None
}

df_pm = (
    df_pm
    .replace(dict_values_to_replace, subset=None)
    .withColumn("Fase", when(col("Fase") == lit("ALFA"), col("Fase")).otherwise(regexp_replace("Fase", r"[^0-9]", "")))
    .withColumn("IPS", when(col("IPS").contains("%"), regexp_replace("IPS", "%", "")/100).otherwise(translate(col("IPS"), ",", ".")))
    .withColumn("Inde", when(col("ano") == lit("2022-01-01"), col("INDE_22")).when(col("ano") == lit("2023-01-01"), col("INDE_23")).otherwise(col("INDE_24")))
    .withColumn("Pedra", when(col("ano") == lit("2022-01-01"), col("Pedra_22")).when(col("ano") == lit("2023-01-01"), col("Pedra_23")).otherwise(col("Pedra_24")))
)

df_pm.limit(5).display()

RA,Fase,Turma,Nome_Anonimizado,Idade,Genero,Ano_ingresso,Instituicao_de_ensino,Pedra_20,Pedra_21,Pedra_22,INDE_22,Cg,Cf,Ct,No_Av,Avaliador1,Rec_Av1,Avaliador2,Rec_Av2,Avaliador3,Rec_Av3,Avaliador4,Rec_Av4,IAA,IEG,IPS,Rec_Psicologia,IDA,Mat,Por,Ing,Indicado,Atingiu_PV,IPV,IAN,Fase_ideal,Defasagem,Destaque_IEG,Destaque_IDA,Destaque_IPV,Ano,Data_de_Nasc,INDE_23,Pedra_23,IPP,INDE_24,Pedra_24,Avaliador5,Avaliador6,Escola,Ativo_Inativo,Inde,Pedra
RA-1,7,A,Aluno-1,19,Feminino,2016,Escola Pública,Ametista,Ametista,Quartzo,"5,783",753,18,10,4,Avaliador-5,Mantido na Fase atual,Avaliador-27,Promovido de Fase + Bolsa,Avaliador-28,Promovido de Fase,Avaliador-31,Mantido na Fase atual,"8,3","4,1",5.6,Requer avaliação,"4,0","2,7","3,5","6,0",Sim,Não,"7,278","5,000",Fase 8 (Universitários),-1,Melhorar: Melhorar a sua entrega de lições de casa.,Melhorar: Empenhar-se mais nas aulas e avaliações.,Melhorar: Integrar-se mais aos Princípios Passos Mágicos.,2022-01-01,2003-01-01,null,null,null,null,null,null,null,null,null,"5,783",Quartzo
RA-2,7,A,Aluno-2,17,Feminino,2017,Rede Decisão,Ametista,Ametista,Ametista,"7,055",469,8,3,4,Avaliador-5,Promovido de Fase,Avaliador-27,Promovido de Fase + Bolsa,Avaliador-28,Promovido de Fase,Avaliador-31,Promovido de Fase + Bolsa,"8,8","5,2",6.3,Sem limitações,"6,8","6,3","4,5","9,7",Não,Não,"6,778","10,000",Fase 7 (3º EM),0,Melhorar: Melhorar a sua entrega de lições de casa.,Melhorar: Empenhar-se mais nas aulas e avaliações.,Melhorar: Integrar-se mais aos Princípios Passos Mágicos.,2022-01-01,2005-01-01,null,null,null,null,null,null,null,null,null,"7,055",Ametista
RA-3,7,A,Aluno-3,17,Feminino,2016,Rede Decisão,Ametista,Ametista,Ágata,"6,591",629,13,6,4,Avaliador-5,Promovido de Fase,Avaliador-27,Promovido de Fase + Bolsa,Avaliador-28,Promovido de Fase + Bolsa,Avaliador-31,Promovido de Fase + Bolsa,"0,0","7,9",5.6,Sem limitações,"5,6","5,8","4,0","6,9",Não,Não,"7,556","10,000",Fase 7 (3º EM),0,Destaque: A sua boa entrega das lições de casa.,Melhorar: Empenhar-se mais nas aulas e avaliações.,Destaque: A sua boa integração aos Princípios Passos Mágicos.,2022-01-01,2005-01-01,null,null,null,null,null,null,null,null,null,"6,591",Ágata
RA-4,7,A,Aluno-4,17,Masculino,2017,Rede Decisão,Ametista,Ametista,Quartzo,"5,951",731,15,7,4,Avaliador-5,Promovido de Fase,Avaliador-27,Mantido na Fase atual,Avaliador-28,Mantido na Fase atual,Avaliador-31,Mantido na Fase atual,"8,8","4,5",5.6,Requer avaliação,"5,0","2,8","3,5","8,7",Não,Não,"5,278","10,000",Fase 7 (3º EM),0,Melhorar: Melhorar a sua entrega de lições de casa.,Melhorar: Empenhar-se mais nas aulas e avaliações.,Melhorar: Integrar-se mais aos Princípios Passos Mágicos.,2022-01-01,2005-01-01,null,null,null,null,null,null,null,null,null,"5,951",Quartzo
RA-5,7,A,Aluno-5,17,Feminino,2016,Rede Decisão,Ametista,Ametista,Ametista,"7,427",344,6,2,4,Avaliador-5,Promovido de Fase,Avaliador-27,Promovido de Fase + Bolsa,Avaliador-28,Promovido de Fase + Bolsa,Avaliador-31,Promovido de Fase + Bolsa,"7,9","8,6",5.6,Requer avaliação,"5,2","7,0","2,9","5,7",Não,Não,"7,389","10,000",Fase 7 (3º EM),0,Destaque: A sua boa entrega das lições de casa.,Melhorar: Empenhar-se mais nas aulas e avaliações.,Melhorar: Integrar-se mais aos Princípios Passos Mágicos.,2022-01-01,2005-01-01,null,null,null,null,null,null,null,null,null,"7,427",Ametista


In [0]:
list_int_cols = [
    "Idade"
    ,"Ano_ingresso"
    ,"Cg"
    ,"Cf"
    ,"Ct"
    ,"No_Av"
    ,"Defasagem"
]

list_decimal_cols = [
    "INDE_22"
    ,"INDE_23"
    ,"INDE_24"
    ,"INDE"
    ,"IAA"
    ,"IEG"
    ,"IPS"
    ,"IDA"
    ,"Mat"
    ,"Por"
    ,"Ing"
    ,"IPV"
    ,"IAN"
    ,"IPP"
]

df_pm = (
    df_pm
    .withColumns({c: col(c).cast("int") for c in list_int_cols})
    .withColumns({c: translate(col(c), ",", ".").cast("decimal(18,4)") for c in list_decimal_cols})
)

df_pm.limit(5).display()

RA,Fase,Turma,Nome_Anonimizado,Idade,Genero,Ano_ingresso,Instituicao_de_ensino,Pedra_20,Pedra_21,Pedra_22,INDE_22,Cg,Cf,Ct,No_Av,Avaliador1,Rec_Av1,Avaliador2,Rec_Av2,Avaliador3,Rec_Av3,Avaliador4,Rec_Av4,IAA,IEG,IPS,Rec_Psicologia,IDA,Mat,Por,Ing,Indicado,Atingiu_PV,IPV,IAN,Fase_ideal,Defasagem,Destaque_IEG,Destaque_IDA,Destaque_IPV,Ano,Data_de_Nasc,INDE_23,Pedra_23,IPP,INDE_24,Pedra_24,Avaliador5,Avaliador6,Escola,Ativo_Inativo,INDE,Pedra
RA-1,7,A,Aluno-1,19,Feminino,2016,Escola Pública,Ametista,Ametista,Quartzo,5.7830,753,18,10,4,Avaliador-5,Mantido na Fase atual,Avaliador-27,Promovido de Fase + Bolsa,Avaliador-28,Promovido de Fase,Avaliador-31,Mantido na Fase atual,8.3000,4.1000,5.6000,Requer avaliação,4.0000,2.7000,3.5000,6.0000,Sim,Não,7.2780,5.0000,Fase 8 (Universitários),-1,Melhorar: Melhorar a sua entrega de lições de casa.,Melhorar: Empenhar-se mais nas aulas e avaliações.,Melhorar: Integrar-se mais aos Princípios Passos Mágicos.,2022-01-01,2003-01-01,null,null,null,null,null,null,null,null,null,5.7830,Quartzo
RA-2,7,A,Aluno-2,17,Feminino,2017,Rede Decisão,Ametista,Ametista,Ametista,7.0550,469,8,3,4,Avaliador-5,Promovido de Fase,Avaliador-27,Promovido de Fase + Bolsa,Avaliador-28,Promovido de Fase,Avaliador-31,Promovido de Fase + Bolsa,8.8000,5.2000,6.3000,Sem limitações,6.8000,6.3000,4.5000,9.7000,Não,Não,6.7780,10.0000,Fase 7 (3º EM),0,Melhorar: Melhorar a sua entrega de lições de casa.,Melhorar: Empenhar-se mais nas aulas e avaliações.,Melhorar: Integrar-se mais aos Princípios Passos Mágicos.,2022-01-01,2005-01-01,null,null,null,null,null,null,null,null,null,7.0550,Ametista
RA-3,7,A,Aluno-3,17,Feminino,2016,Rede Decisão,Ametista,Ametista,Ágata,6.5910,629,13,6,4,Avaliador-5,Promovido de Fase,Avaliador-27,Promovido de Fase + Bolsa,Avaliador-28,Promovido de Fase + Bolsa,Avaliador-31,Promovido de Fase + Bolsa,0.0000,7.9000,5.6000,Sem limitações,5.6000,5.8000,4.0000,6.9000,Não,Não,7.5560,10.0000,Fase 7 (3º EM),0,Destaque: A sua boa entrega das lições de casa.,Melhorar: Empenhar-se mais nas aulas e avaliações.,Destaque: A sua boa integração aos Princípios Passos Mágicos.,2022-01-01,2005-01-01,null,null,null,null,null,null,null,null,null,6.5910,Ágata
RA-4,7,A,Aluno-4,17,Masculino,2017,Rede Decisão,Ametista,Ametista,Quartzo,5.9510,731,15,7,4,Avaliador-5,Promovido de Fase,Avaliador-27,Mantido na Fase atual,Avaliador-28,Mantido na Fase atual,Avaliador-31,Mantido na Fase atual,8.8000,4.5000,5.6000,Requer avaliação,5.0000,2.8000,3.5000,8.7000,Não,Não,5.2780,10.0000,Fase 7 (3º EM),0,Melhorar: Melhorar a sua entrega de lições de casa.,Melhorar: Empenhar-se mais nas aulas e avaliações.,Melhorar: Integrar-se mais aos Princípios Passos Mágicos.,2022-01-01,2005-01-01,null,null,null,null,null,null,null,null,null,5.9510,Quartzo
RA-5,7,A,Aluno-5,17,Feminino,2016,Rede Decisão,Ametista,Ametista,Ametista,7.4270,344,6,2,4,Avaliador-5,Promovido de Fase,Avaliador-27,Promovido de Fase + Bolsa,Avaliador-28,Promovido de Fase + Bolsa,Avaliador-31,Promovido de Fase + Bolsa,7.9000,8.6000,5.6000,Requer avaliação,5.2000,7.0000,2.9000,5.7000,Não,Não,7.3890,10.0000,Fase 7 (3º EM),0,Destaque: A sua boa entrega das lições de casa.,Melhorar: Empenhar-se mais nas aulas e avaliações.,Melhorar: Integrar-se mais aos Princípios Passos Mágicos.,2022-01-01,2005-01-01,null,null,null,null,null,null,null,null,null,7.4270,Ametista


In [0]:
dict_defasagem = {
    10: "Em fase"
    ,5: "Moderada"
    ,2.5: "Severa"
}

# Map para criar coluna de defasagem com base no dicionario
mapping_expr = create_map([lit(x) for x in chain(*dict_defasagem.items())])

df_pm = (
    df_pm
    .withColumn("ID", regexp_replace(col("RA"), "^.{3}", "").cast("int"))
    # Aplicar o mapeamento na nova coluna (retorna NULL se nao mapeado)
    .withColumn("Ian_defasagem", mapping_expr[col("ian")])
)

df_pm.limit(5).display()

RA,Fase,Turma,Nome_Anonimizado,Idade,Genero,Ano_ingresso,Instituicao_de_ensino,Pedra_20,Pedra_21,Pedra_22,INDE_22,Cg,Cf,Ct,No_Av,Avaliador1,Rec_Av1,Avaliador2,Rec_Av2,Avaliador3,Rec_Av3,Avaliador4,Rec_Av4,IAA,IEG,IPS,Rec_Psicologia,IDA,Mat,Por,Ing,Indicado,Atingiu_PV,IPV,IAN,Fase_ideal,Defasagem,Destaque_IEG,Destaque_IDA,Destaque_IPV,Ano,Data_de_Nasc,INDE_23,Pedra_23,IPP,INDE_24,Pedra_24,Avaliador5,Avaliador6,Escola,Ativo_Inativo,INDE,Pedra,ID,Ian_defasagem
RA-1,7,A,Aluno-1,19,Feminino,2016,Escola Pública,Ametista,Ametista,Quartzo,5.7830,753,18,10,4,Avaliador-5,Mantido na Fase atual,Avaliador-27,Promovido de Fase + Bolsa,Avaliador-28,Promovido de Fase,Avaliador-31,Mantido na Fase atual,8.3000,4.1000,5.6000,Requer avaliação,4.0000,2.7000,3.5000,6.0000,Sim,Não,7.2780,5.0000,Fase 8 (Universitários),-1,Melhorar: Melhorar a sua entrega de lições de casa.,Melhorar: Empenhar-se mais nas aulas e avaliações.,Melhorar: Integrar-se mais aos Princípios Passos Mágicos.,2022-01-01,2003-01-01,null,null,null,null,null,null,null,null,null,5.7830,Quartzo,1,Moderada
RA-2,7,A,Aluno-2,17,Feminino,2017,Rede Decisão,Ametista,Ametista,Ametista,7.0550,469,8,3,4,Avaliador-5,Promovido de Fase,Avaliador-27,Promovido de Fase + Bolsa,Avaliador-28,Promovido de Fase,Avaliador-31,Promovido de Fase + Bolsa,8.8000,5.2000,6.3000,Sem limitações,6.8000,6.3000,4.5000,9.7000,Não,Não,6.7780,10.0000,Fase 7 (3º EM),0,Melhorar: Melhorar a sua entrega de lições de casa.,Melhorar: Empenhar-se mais nas aulas e avaliações.,Melhorar: Integrar-se mais aos Princípios Passos Mágicos.,2022-01-01,2005-01-01,null,null,null,null,null,null,null,null,null,7.0550,Ametista,2,Em fase
RA-3,7,A,Aluno-3,17,Feminino,2016,Rede Decisão,Ametista,Ametista,Ágata,6.5910,629,13,6,4,Avaliador-5,Promovido de Fase,Avaliador-27,Promovido de Fase + Bolsa,Avaliador-28,Promovido de Fase + Bolsa,Avaliador-31,Promovido de Fase + Bolsa,0.0000,7.9000,5.6000,Sem limitações,5.6000,5.8000,4.0000,6.9000,Não,Não,7.5560,10.0000,Fase 7 (3º EM),0,Destaque: A sua boa entrega das lições de casa.,Melhorar: Empenhar-se mais nas aulas e avaliações.,Destaque: A sua boa integração aos Princípios Passos Mágicos.,2022-01-01,2005-01-01,null,null,null,null,null,null,null,null,null,6.5910,Ágata,3,Em fase
RA-4,7,A,Aluno-4,17,Masculino,2017,Rede Decisão,Ametista,Ametista,Quartzo,5.9510,731,15,7,4,Avaliador-5,Promovido de Fase,Avaliador-27,Mantido na Fase atual,Avaliador-28,Mantido na Fase atual,Avaliador-31,Mantido na Fase atual,8.8000,4.5000,5.6000,Requer avaliação,5.0000,2.8000,3.5000,8.7000,Não,Não,5.2780,10.0000,Fase 7 (3º EM),0,Melhorar: Melhorar a sua entrega de lições de casa.,Melhorar: Empenhar-se mais nas aulas e avaliações.,Melhorar: Integrar-se mais aos Princípios Passos Mágicos.,2022-01-01,2005-01-01,null,null,null,null,null,null,null,null,null,5.9510,Quartzo,4,Em fase
RA-5,7,A,Aluno-5,17,Feminino,2016,Rede Decisão,Ametista,Ametista,Ametista,7.4270,344,6,2,4,Avaliador-5,Promovido de Fase,Avaliador-27,Promovido de Fase + Bolsa,Avaliador-28,Promovido de Fase + Bolsa,Avaliador-31,Promovido de Fase + Bolsa,7.9000,8.6000,5.6000,Requer avaliação,5.2000,7.0000,2.9000,5.7000,Não,Não,7.3890,10.0000,Fase 7 (3º EM),0,Destaque: A sua boa entrega das lições de casa.,Melhorar: Empenhar-se mais nas aulas e avaliações.,Melhorar: Integrar-se mais aos Princípios Passos Mágicos.,2022-01-01,2005-01-01,null,null,null,null,null,null,null,null,null,7.4270,Ametista,5,Em fase


In [0]:
cols_ordenadas = [
    "ID"
    ,"RA"
    ,"Ano"
    ,"Nome_Anonimizado"
    ,"Genero"
    ,"Data_de_Nasc"
    ,"Idade"
    ,"Fase"
    ,"Fase_ideal"
    ,"Defasagem"
    ,"Turma"
    ,"Escola"
    ,"Instituicao_de_ensino"
    ,"Ano_ingresso"
    ,"Pedra_20"
    ,"Pedra_21"
    ,"Pedra_22"
    ,"Pedra_23"
    ,"Pedra_24"
    ,"Pedra"
    ,"INDE_22"
    ,"INDE_23"
    ,"INDE_24"
    ,"INDE"
    ,"Rec_Av1"
    ,"Rec_Av2"
    ,"Rec_Av3"
    ,"Rec_Av4"
    ,"No_Av"
    ,"Avaliador1"
    ,"Avaliador2"
    ,"Avaliador3"
    ,"Avaliador4"
    ,"Avaliador5"
    ,"Avaliador6"
    ,"Rec_Psicologia"
    ,"IAN"
    ,"Ian_defasagem"
    ,"IDA"
    ,"IEG"
    ,"IAA"
    ,"IPS"
    ,"IPP"
    ,"IPV"
    ,"Atingiu_PV"
    ,"Mat"
    ,"Por"
    ,"Ing"
    ,"Destaque_IEG"
    ,"Destaque_IDA"
    ,"Destaque_IPV"
    ,"Indicado"
    ,"Cg"
    ,"Cf"
    ,"Ct"
    ,"Ativo_Inativo"
]

# Ondenacao manual para facilitar leitura dos dados
df_pm = df_pm.select(cols_ordenadas)

df_pm.limit(5).display()

ID,RA,Ano,Nome_Anonimizado,Genero,Data_de_Nasc,Idade,Fase,Fase_ideal,Defasagem,Turma,Escola,Instituicao_de_ensino,Ano_ingresso,Pedra_20,Pedra_21,Pedra_22,Pedra_23,Pedra_24,Pedra,INDE_22,INDE_23,INDE_24,INDE,Rec_Av1,Rec_Av2,Rec_Av3,Rec_Av4,No_Av,Avaliador1,Avaliador2,Avaliador3,Avaliador4,Avaliador5,Avaliador6,Rec_Psicologia,IAN,Ian_defasagem,IDA,IEG,IAA,IPS,IPP,IPV,Atingiu_PV,Mat,Por,Ing,Destaque_IEG,Destaque_IDA,Destaque_IPV,Indicado,Cg,Cf,Ct,Ativo_Inativo
1,RA-1,2022-01-01,Aluno-1,Feminino,2003-01-01,19,7,Fase 8 (Universitários),-1,A,null,Escola Pública,2016,Ametista,Ametista,Quartzo,null,null,Quartzo,5.7830,null,null,5.7830,Mantido na Fase atual,Promovido de Fase + Bolsa,Promovido de Fase,Mantido na Fase atual,4,Avaliador-5,Avaliador-27,Avaliador-28,Avaliador-31,null,null,Requer avaliação,5.0000,Moderada,4.0000,4.1000,8.3000,5.6000,null,7.2780,Não,2.7000,3.5000,6.0000,Melhorar: Melhorar a sua entrega de lições de casa.,Melhorar: Empenhar-se mais nas aulas e avaliações.,Melhorar: Integrar-se mais aos Princípios Passos Mágicos.,Sim,753,18,10,null
2,RA-2,2022-01-01,Aluno-2,Feminino,2005-01-01,17,7,Fase 7 (3º EM),0,A,null,Rede Decisão,2017,Ametista,Ametista,Ametista,null,null,Ametista,7.0550,null,null,7.0550,Promovido de Fase,Promovido de Fase + Bolsa,Promovido de Fase,Promovido de Fase + Bolsa,4,Avaliador-5,Avaliador-27,Avaliador-28,Avaliador-31,null,null,Sem limitações,10.0000,Em fase,6.8000,5.2000,8.8000,6.3000,null,6.7780,Não,6.3000,4.5000,9.7000,Melhorar: Melhorar a sua entrega de lições de casa.,Melhorar: Empenhar-se mais nas aulas e avaliações.,Melhorar: Integrar-se mais aos Princípios Passos Mágicos.,Não,469,8,3,null
3,RA-3,2022-01-01,Aluno-3,Feminino,2005-01-01,17,7,Fase 7 (3º EM),0,A,null,Rede Decisão,2016,Ametista,Ametista,Ágata,null,null,Ágata,6.5910,null,null,6.5910,Promovido de Fase,Promovido de Fase + Bolsa,Promovido de Fase + Bolsa,Promovido de Fase + Bolsa,4,Avaliador-5,Avaliador-27,Avaliador-28,Avaliador-31,null,null,Sem limitações,10.0000,Em fase,5.6000,7.9000,0.0000,5.6000,null,7.5560,Não,5.8000,4.0000,6.9000,Destaque: A sua boa entrega das lições de casa.,Melhorar: Empenhar-se mais nas aulas e avaliações.,Destaque: A sua boa integração aos Princípios Passos Mágicos.,Não,629,13,6,null
4,RA-4,2022-01-01,Aluno-4,Masculino,2005-01-01,17,7,Fase 7 (3º EM),0,A,null,Rede Decisão,2017,Ametista,Ametista,Quartzo,null,null,Quartzo,5.9510,null,null,5.9510,Promovido de Fase,Mantido na Fase atual,Mantido na Fase atual,Mantido na Fase atual,4,Avaliador-5,Avaliador-27,Avaliador-28,Avaliador-31,null,null,Requer avaliação,10.0000,Em fase,5.0000,4.5000,8.8000,5.6000,null,5.2780,Não,2.8000,3.5000,8.7000,Melhorar: Melhorar a sua entrega de lições de casa.,Melhorar: Empenhar-se mais nas aulas e avaliações.,Melhorar: Integrar-se mais aos Princípios Passos Mágicos.,Não,731,15,7,null
5,RA-5,2022-01-01,Aluno-5,Feminino,2005-01-01,17,7,Fase 7 (3º EM),0,A,null,Rede Decisão,2016,Ametista,Ametista,Ametista,null,null,Ametista,7.4270,null,null,7.4270,Promovido de Fase,Promovido de Fase + Bolsa,Promovido de Fase + Bolsa,Promovido de Fase + Bolsa,4,Avaliador-5,Avaliador-27,Avaliador-28,Avaliador-31,null,null,Requer avaliação,10.0000,Em fase,5.2000,8.6000,7.9000,5.6000,null,7.3890,Não,7.0000,2.9000,5.7000,Destaque: A sua boa entrega das lições de casa.,Melhorar: Empenhar-se mais nas aulas e avaliações.,Melhorar: Integrar-se mais aos Princípios Passos Mágicos.,Não,344,6,2,null


## Carga

In [0]:
df_pm.write.mode("overwrite").saveAsTable("pos_fiap.datathon.passos_magicos")

# Exploração

In [0]:
df = spark.table("pos_fiap.datathon.passos_magicos")

In [0]:
print((df.count(), len(df.columns)))

(3030, 56)


In [0]:
df.select([count(when(col(c).isNull(), c)).alias(c) for c in df.columns]).display()

RA,Fase,Turma,Nome_Anonimizado,Idade,Genero,Ano_ingresso,Instituicao_de_ensino,Pedra_20,Pedra_21,Pedra_22,INDE_22,Cg,Cf,Ct,No_Av,Avaliador1,Rec_Av1,Avaliador2,Rec_Av2,Avaliador3,Rec_Av3,Avaliador4,Rec_Av4,IAA,IEG,IPS,Rec_Psicologia,IDA,Mat,Por,Ing,Indicado,Atingiu_PV,IPV,IAN,Fase_ideal,Defasagem,Destaque_IEG,Destaque_IDA,Destaque_IPV,Ano,Data_de_Nasc,INDE_23,Pedra_23,IPP,INDE_24,Pedra_24,Avaliador5,Avaliador6,Escola,Ativo_Inativo,ID,INDE,Pedra,ian_defasagem
0,0,0,0,0,0,0,1,2276,1969,1098,1098,2170,2170,2170,76,203,2170,203,2170,996,2170,1979,2734,165,76,171,2170,178,184,185,1939,2170,2170,178,0,0,0,2170,2170,2170,0,0,1409,1409,1038,1976,1976,2882,3024,1875,1874,0,185,185,0


In [0]:
(
    df
    .groupBy("ano")
    .agg(
        count("id").alias("count")
        ,countDistinct("id").alias("count_distinct")
        ,min("id").alias("min")
        ,max("id").alias("max")
    )
    .orderBy("ano")
).display()

ano,count,count_distinct,min,max
2022-01-01,860,860,1,860
2023-01-01,1014,1014,1,1274
2024-01-01,1156,1156,1,1661
